# Пункт 3. Проверка LoRA-обучения Qwen3-4B на T4

По оценкам Qwen3.8-27B и Gemma-4-31B-it выбран
`Qwen/Qwen3-4B-Instruct-2507` с промптом `household`.
Этот запуск проверит обучение на T4, маску loss, нулевой вклад свежей LoRA,
один шаг оптимизатора и сохранение / загрузку адаптера.

Откройте **новую сессию Colab T4 GPU** и выполняйте ячейки по порядку.
Все настройки уже выбраны. В тесте четыре коротких арифметических примера,
длина тензоров 768 токенов, batch 1 и накопление градиента 4.
Получится один шаг AdamW с LR 5e-5 и LoRA r16 / alpha32 на семи проекциях.

Результат этого пункта — технические проверки и адаптер после контрольного шага.
Оценка улучшения качества выполняется после подготовки проверенных данных и SFT-пилота.

[Инструкция](https://github.com/maria-igntv/lm-hw/blob/main/docs/sft_v3_smoke_guide.md).


In [ ]:
import importlib.metadata
import json
import os
import subprocess
import sys
from pathlib import Path

ROOT = Path("/content/lm-hw-v3")
if not ROOT.exists():
    subprocess.run(["git", "clone", "https://github.com/maria-igntv/lm-hw.git", str(ROOT)], check=True)
else:
    subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
smoke_config = json.loads((ROOT / "data/sft_v3/smoke_config.json").read_text(encoding="utf-8"))
selection = json.loads((ROOT / smoke_config["selection_path"]).read_text(encoding="utf-8"))
requirements = [f"{package}=={version}" for package, version in smoke_config["packages"].items()]
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *requirements], check=True)
for package, version in smoke_config["packages"].items():
    loaded = sys.modules.get(package.replace("-", "_"))
    if loaded is not None and getattr(loaded, "__version__", version) != version:
        raise RuntimeError("Импортирована другая версия библиотеки. Перезапустите сеанс Python и выполните ячейки заново.")
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
print("Выбрана модель", selection["model_id"])
print("Ревизия", selection["revision"])
print("Выбран промпт", selection["prompt_id"])


## 1. Фиксируем среду и данные

Здесь создаётся папка результата и журнал ошибок. Четыре контрольных ответа
проверяются арифметически по своим условиям. Системный промпт взят из выбранной
конфигурации. Входы и код сохраняются для воспроизведения.


In [ ]:
import hashlib
import platform
import re
import shutil
import time
import traceback
from datetime import datetime, timezone

import torch

from checks.screening_v3 import read_jsonl, sha256_file, write_json, write_jsonl
from checks.sft_smoke_v3 import build_masked_example

if not torch.cuda.is_available():
    raise RuntimeError("Выберите Среда выполнения → Сменить среду выполнения → T4 GPU.")
if "smoke_model" in globals():
    raise RuntimeError("Модель этого smoke уже находится в памяти. Для нового полного прогона откройте новую T4-сессию.")
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.backends.cudnn.benchmark = False
rows = read_jsonl(ROOT / "data/sft_v3/smoke_examples.jsonl")
if len(rows) != smoke_config["gradient_accumulation_steps"] or smoke_config["optimizer_steps"] != 1:
    raise ValueError("Тест ожидает четыре микробатча и один шаг оптимизатора")
if any(row["messages"][0]["content"] != selection["system_prompt"] for row in rows):
    raise ValueError("Системный промпт контрольных примеров отличается от выбора")
development_questions = {row["question"].strip().casefold() for row in read_jsonl(ROOT / "data/development_v3/questions.jsonl")}
if any(row["messages"][1]["content"].strip().casefold() in development_questions for row in rows):
    raise ValueError("Контроль обучения совпал с development-вопросом")

RUN_ID = "qwen3_4b_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%f")
OUT_DIR = ROOT / "results/sft_v3_smoke" / RUN_ID
OUT_DIR.mkdir(parents=True, exist_ok=False)
ADAPTER_DIR = OUT_DIR / "adapter"
tracked_files = [
    "data/sft_v3/selection.json", "data/sft_v3/smoke_config.json", "data/sft_v3/smoke_examples.jsonl",
    "checks/sft_smoke_v3.py", "checks/screening_v3.py", "checks/autocheck.py", "checks/evaluation.py",
    "notebooks/02_sft_v3_smoke.ipynb",
]
metadata = {
    "run_id": RUN_ID, "status": "prepared", "smoke_id": smoke_config["smoke_id"],
    "started_at_utc": datetime.now(timezone.utc).isoformat(),
    "selection": selection, "settings": smoke_config,
    "repo_commit": subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip(),
    "file_sha256": {name: sha256_file(ROOT / name) for name in tracked_files},
    "versions": {package: importlib.metadata.version(package) for package in ["torch", *smoke_config["packages"]]},
    "hardware": {
        "gpu": torch.cuda.get_device_name(0), "torch_cuda": torch.version.cuda,
        "python": platform.python_version(),
        "gpu_memory_gib": torch.cuda.get_device_properties(0).total_memory / 1024**3,
        "nvidia_smi": subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"], text=True
        ).strip(),
    },
}
write_json(OUT_DIR / "run_metadata.json", metadata)
write_jsonl(OUT_DIR / "errors.jsonl", [])
for filename in tracked_files:
    destination = OUT_DIR / "reproduction" / filename
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(ROOT / filename, destination)
(OUT_DIR / "pip_freeze.txt").write_text(
    subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True), encoding="utf-8"
)


def record_failure(stage, error):
    errors = read_jsonl(OUT_DIR / "errors.jsonl")
    errors.append({"stage": stage, "type": type(error).__name__, "message": str(error), "traceback": traceback.format_exc()})
    write_jsonl(OUT_DIR / "errors.jsonl", errors)
    metadata["status"] = "failed"
    metadata["failed_stage"] = stage
    write_json(OUT_DIR / "run_metadata.json", metadata)


print(json.dumps(metadata["hardware"], ensure_ascii=False, indent=2))
print("Результаты", OUT_DIR)


## 2. Загружаем модель и проверяем маски

Используются те же ревизия, NF4, FP16 и eager attention, что в скрининге.
Loss учитывает текст ассистента и EOS конца сообщения. Системный промпт,
вопрос и padding получают метку `-100`, которая исключает их из loss.
Тензоры дополняются справа до 768 для проверки памяти на выбранной длине.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, GenerationConfig, set_seed
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training

set_seed(smoke_config["seed"])
try:
    load_started = time.perf_counter()
    tokenizer = AutoTokenizer.from_pretrained(selection["model_id"], revision=selection["revision"], trust_remote_code=False)
    tokenizer.padding_side = "right"
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    quantization = dict(smoke_config["quantization"])
    quantization["bnb_4bit_compute_dtype"] = torch.float16
    smoke_model = AutoModelForCausalLM.from_pretrained(
        selection["model_id"], revision=selection["revision"], trust_remote_code=False,
        dtype=torch.float16, device_map={"": 0},
        quantization_config=BitsAndBytesConfig(**quantization), attn_implementation=smoke_config["attention"],
    ).eval()
    original_generation = smoke_model.generation_config.to_dict()
    if set(original_generation["eos_token_id"]) != {151645, 151643}:
        raise ValueError("Неожиданные EOS выбранной Qwen-модели")
    if not getattr(smoke_model, "is_loaded_in_4bit", False):
        raise ValueError("Не подтверждена 4-битная загрузка")
    if any(parameter.device.type != "cuda" for parameter in smoke_model.parameters()):
        raise ValueError("Часть модели находится вне GPU")
    training_batches = []
    mask_audit = []
    for row in rows:
        batch, audit = build_masked_example(tokenizer, row, smoke_config["max_seq_length"])
        training_batches.append(batch)
        mask_audit.append(audit)
    write_jsonl(OUT_DIR / "mask_audit.jsonl", mask_audit)
    write_jsonl(OUT_DIR / "training_controls.jsonl", rows)
    probe_ids = training_batches[0]["input_ids"][:mask_audit[0]["prompt_tokens"]]
    probe_inputs = {
        "input_ids": torch.tensor([probe_ids], dtype=torch.long, device="cuda"),
        "attention_mask": torch.ones((1, len(probe_ids)), dtype=torch.long, device="cuda"),
    }
    metadata["model_load_seconds"] = time.perf_counter() - load_started
    metadata["model_memory_gib_before_preparation"] = smoke_model.get_memory_footprint() / 1024**3
    metadata["status"] = "model_loaded"
    write_json(OUT_DIR / "run_metadata.json", metadata)
except (Exception, KeyboardInterrupt) as error:
    record_failure("load_and_masks", error)
    raise


@torch.inference_mode()
def last_logits():
    smoke_model.eval()
    logits = smoke_model(**probe_inputs, use_cache=False, logits_to_keep=1).logits[0, -1].float().cpu()
    if not torch.isfinite(logits).all():
        raise FloatingPointError("В контрольных logits появились NaN или Inf")
    return logits


@torch.inference_mode()
def greedy_probe():
    smoke_model.eval()
    generation = GenerationConfig.from_dict({
        **original_generation, "max_new_tokens": 32, "do_sample": False,
        "temperature": 1.0, "top_p": 1.0, "top_k": 0, "min_p": 0.0,
        "num_beams": 1, "use_cache": True, "return_dict_in_generate": False,
    })
    output = smoke_model.generate(**probe_inputs, generation_config=generation)
    token_ids = output[0, len(probe_ids):].tolist()
    return {"token_ids": token_ids, "text": tokenizer.decode(token_ids, skip_special_tokens=True)}


try:
    original_logits = last_logits()
except (Exception, KeyboardInterrupt) as error:
    record_failure("baseline_logits", error)
    raise
print("Маски проверены, примеров", len(mask_audit))
for audit in mask_audit:
    print(audit["sample_id"], "обучаемых токенов", audit["supervised_tokens"], "EOS", audit["last_supervised_token_id"])
    print(audit["supervised_text"])


## 3. Подключаем свежую LoRA и проверяем её нулевой вклад

Подготовка к k-bit-обучению замораживает базовые веса, включает checkpointing
и переводит часть параметров в float32. Её влияние на logits сохраняется отдельно.
Затем сравниваются включённая свежая LoRA и отключённая LoRA в подготовленной модели.
У свежего адаптера матрицы B должны быть нулевыми, а logits совпадать.


In [ ]:
try:
    smoke_model = prepare_model_for_kbit_training(
        smoke_model, use_gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
    )
    smoke_model.config.use_cache = False
    prepared_logits = last_logits()
    smoke_model = get_peft_model(smoke_model, LoraConfig(**smoke_config["lora"]))
    smoke_model.eval()
    active_logits = last_logits()
    with smoke_model.disable_adapter():
        disabled_logits = last_logits()
    b_matrices = [(name, parameter) for name, parameter in smoke_model.named_parameters() if "lora_B" in name]
    zero_b = bool(b_matrices) and all(torch.count_nonzero(parameter).item() == 0 for _, parameter in b_matrices)
    identity = {
        "all_B_zero": zero_b,
        "B_matrices": len(b_matrices),
        "preparation_logit_max_abs_diff": float((original_logits - prepared_logits).abs().max()),
        "active_vs_disabled_max_abs_diff": float((active_logits - disabled_logits).abs().max()),
        "active_vs_disabled_allclose": torch.allclose(active_logits, disabled_logits, rtol=1e-5, atol=1e-5),
        "trainable_parameters": sum(parameter.numel() for parameter in smoke_model.parameters() if parameter.requires_grad),
        "gradient_checkpointing": smoke_model.is_gradient_checkpointing,
        "base_weights_frozen": all(not parameter.requires_grad for name, parameter in smoke_model.named_parameters() if "lora_" not in name),
    }
    write_json(OUT_DIR / "fresh_adapter_check.json", identity)
    if not identity["all_B_zero"] or not identity["active_vs_disabled_allclose"]:
        raise ValueError("Свежая LoRA не прошла проверку нулевого вклада")
    if not identity["trainable_parameters"] or not identity["gradient_checkpointing"] or not identity["base_weights_frozen"]:
        raise ValueError("Нарушены условия trainable-параметров, заморозки базы или checkpointing")
    del b_matrices
    metadata["status"] = "fresh_adapter_verified"
    write_json(OUT_DIR / "run_metadata.json", metadata)
except (Exception, KeyboardInterrupt) as error:
    record_failure("fresh_adapter_identity", error)
    raise

print(json.dumps(identity, ensure_ascii=False, indent=2))


## 4. Четыре микробатча и один шаг AdamW

Каждый пример обрабатывается отдельно. Градиенты накапливаются четыре раза,
затем выполняется один шаг оптимизатора. Проверяются конечные loss и градиенты,
изменение матриц B и пик памяти. Для этого технического теста снижение loss
после одного шага выводится как наблюдение. Критериями завершения служат
численная корректность и фактическое обновление адаптера.


In [ ]:
if globals().get("optimizer_step_done", False):
    raise RuntimeError("Контрольный шаг уже выполнен. Продолжайте к сохранению и проверке загрузки.")
try:
    trainable_parameters = [parameter for parameter in smoke_model.parameters() if parameter.requires_grad]
    if any(parameter.dtype != torch.float32 for parameter in trainable_parameters):
        raise ValueError("Для GradScaler ожидаются float32-параметры адаптера")
    optimizer = torch.optim.AdamW(
        trainable_parameters, lr=smoke_config["learning_rate"],
        weight_decay=smoke_config["weight_decay"], foreach=False,
    )
    scaler = torch.amp.GradScaler("cuda", init_scale=128.0)
    optimizer.zero_grad(set_to_none=True)
    smoke_model.train()
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    training_started = time.perf_counter()
    loss_rows = []
    for index, (batch, audit) in enumerate(zip(training_batches, mask_audit), 1):
        tensors = {key: torch.tensor([value], dtype=torch.long, device="cuda") for key, value in batch.items()}
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            result = smoke_model(**tensors, use_cache=False)
            loss = result.loss
        if not torch.isfinite(loss):
            raise FloatingPointError("Loss содержит NaN или Inf")
        scaler.scale(loss / smoke_config["gradient_accumulation_steps"]).backward()
        loss_rows.append({"microbatch": index, "sample_id": audit["sample_id"], "loss": float(loss.detach().cpu())})
        write_jsonl(OUT_DIR / "training_losses.jsonl", loss_rows)
        print(index, audit["sample_id"], "loss", loss_rows[-1]["loss"])
        del result, loss, tensors

    scaler.unscale_(optimizer)
    parameters_with_grad = [parameter for parameter in trainable_parameters if parameter.grad is not None]
    if not parameters_with_grad or not all(torch.isfinite(parameter.grad).all().item() for parameter in parameters_with_grad):
        raise FloatingPointError("Градиенты отсутствуют или содержат NaN / Inf")
    gradient_norm = torch.nn.utils.clip_grad_norm_(
        trainable_parameters, smoke_config["max_grad_norm"], error_if_nonfinite=True,
    )
    scaler.step(optimizer)
    scaler.update()
    optimizer_step_done = True
    optimizer.zero_grad(set_to_none=True)
    torch.cuda.synchronize()
    nonzero_b = sum(torch.count_nonzero(parameter).item() > 0
                    for name, parameter in smoke_model.named_parameters() if "lora_B" in name)
    training_summary = {
        "optimizer_steps": 1, "microbatches": len(loss_rows),
        "mean_training_loss": sum(row["loss"] for row in loss_rows) / len(loss_rows),
        "grad_norm_before_clip": float(gradient_norm.detach().cpu()),
        "parameters_with_grad": len(parameters_with_grad), "nonzero_B_matrices_after_step": nonzero_b,
        "scaler_scale_after_step": scaler.get_scale(),
        "training_seconds": time.perf_counter() - training_started,
        "peak_allocated_gib": torch.cuda.max_memory_allocated() / 1024**3,
        "peak_reserved_gib": torch.cuda.max_memory_reserved() / 1024**3,
        "sequence_length": smoke_config["max_seq_length"], "batch_size": 1,
    }
    write_json(OUT_DIR / "training_summary.json", training_summary)
    if not nonzero_b:
        raise ValueError("После шага оптимизатора матрицы B остались нулевыми")
    smoke_model.eval()
    first_batch = {key: torch.tensor([value], dtype=torch.long, device="cuda") for key, value in training_batches[0].items()}
    with torch.inference_mode(), torch.autocast(device_type="cuda", dtype=torch.float16):
        loss_after = smoke_model(**first_batch, use_cache=False).loss
    if not torch.isfinite(loss_after):
        raise FloatingPointError("Loss после обновления содержит NaN / Inf")
    training_summary["first_example_loss_before"] = loss_rows[0]["loss"]
    training_summary["first_example_loss_after"] = float(loss_after.cpu())
    write_json(OUT_DIR / "training_summary.json", training_summary)
    del first_batch, loss_after, optimizer, scaler, parameters_with_grad, trainable_parameters
    metadata["status"] = "optimizer_step_verified"
    write_json(OUT_DIR / "run_metadata.json", metadata)
except (Exception, KeyboardInterrupt) as error:
    record_failure("optimizer_step", error)
    raise

print(json.dumps(training_summary, ensure_ascii=False, indent=2))


## 5. Сохраняем safetensors и проверяем повторную загрузку

Сохраняются LoRA-веса и токенизатор. Затем адаптер выгружается из той же базовой
модели и загружается из сохранённой папки. Сравниваются контрольные logits и
greedy-токены до и после загрузки. Базовые веса остаются загруженными на GPU,
поэтому повторное скачивание полной модели для этой проверки не требуется.


In [ ]:
from safetensors import safe_open

try:
    smoke_model.eval()
    trained_logits = last_logits()
    generation_before_reload = greedy_probe()
    ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
    smoke_model.peft_config["default"].base_model_name_or_path = selection["model_id"]
    smoke_model.peft_config["default"].revision = selection["revision"]
    smoke_model.save_pretrained(ADAPTER_DIR, safe_serialization=True, save_embedding_layers=False)
    tokenizer.save_pretrained(ADAPTER_DIR)
    weights_path = ADAPTER_DIR / "adapter_model.safetensors"
    tensor_count = 0
    with safe_open(weights_path, framework="pt", device="cpu") as saved:
        for key in saved.keys():
            value = saved.get_tensor(key)
            if not torch.isfinite(value).all():
                raise FloatingPointError("В сохранённых весах есть NaN / Inf")
            tensor_count += 1
    base_for_reload = smoke_model.unload()
    del smoke_model
    smoke_model = PeftModel.from_pretrained(base_for_reload, ADAPTER_DIR, is_trainable=False).eval()
    smoke_model.config.use_cache = False
    reloaded_logits = last_logits()
    generation_after_reload = greedy_probe()
    roundtrip = {
        "adapter_file": weights_path.name, "adapter_sha256": sha256_file(weights_path),
        "adapter_bytes": weights_path.stat().st_size, "saved_tensors": tensor_count,
        "logits_max_abs_diff": float((trained_logits - reloaded_logits).abs().max()),
        "logits_allclose": torch.allclose(trained_logits, reloaded_logits, rtol=1e-4, atol=1e-4),
        "greedy_tokens_equal": generation_before_reload["token_ids"] == generation_after_reload["token_ids"],
        "generation_before_reload": generation_before_reload,
        "generation_after_reload": generation_after_reload,
    }
    write_json(OUT_DIR / "save_reload_check.json", roundtrip)
    if not roundtrip["logits_allclose"] or not roundtrip["greedy_tokens_equal"]:
        raise ValueError("Адаптер после загрузки даёт другой контрольный результат")
    metadata["status"] = "complete"
    metadata["finished_at_utc"] = datetime.now(timezone.utc).isoformat()
    metadata["adapter_sha256"] = roundtrip["adapter_sha256"]
    write_json(OUT_DIR / "run_metadata.json", metadata)
    write_json(OUT_DIR / "checks.json", {
        "status": "PASS", "mask_examples": len(mask_audit),
        "fresh_adapter_identity": identity["active_vs_disabled_allclose"],
        "base_weights_frozen": identity["base_weights_frozen"],
        "optimizer_steps": training_summary["optimizer_steps"],
        "nonzero_B_matrices": training_summary["nonzero_B_matrices_after_step"],
        "saved_weights_finite": True, "reload_logits_match": roundtrip["logits_allclose"],
        "reload_greedy_tokens_match": roundtrip["greedy_tokens_equal"],
    })
except (Exception, KeyboardInterrupt) as error:
    record_failure("save_and_reload", error)
    raise

print("SMOKE PASS")
print("Пик выделенной памяти обучения, GiB", training_summary["peak_allocated_gib"])
print("SHA-256 адаптера", roundtrip["adapter_sha256"])
print("Logits после загрузки совпали", roundtrip["logits_allclose"])
print("Greedy-токены после загрузки совпали", roundtrip["greedy_tokens_equal"])


## 6. Скачиваем результат

Эту ячейку можно запустить после ошибки, чтобы сохранить уже созданные логи.
Архив содержит `checks.json` при полном успехе, сводку обучения, маски, контроль
подключения / загрузки и папку `adapter` с safetensors.

Дополнительно сохраните выполненный IPYNB через **Файл → Скачать → Скачать файл IPYNB**.
Для анализа понадобятся ZIP и выполненный ноутбук. После проверки результатов
следующим пунктом будет подготовка 80–120 проверенных обучающих ответов.


In [ ]:
from google.colab import files

archive = shutil.make_archive(
    str(Path("/content") / ("sft_v3_smoke_" + RUN_ID)),
    "zip", root_dir=OUT_DIR.parent, base_dir=OUT_DIR.name,
)
files.download(archive)
print("Скачивается", archive)
print("Статус", json.loads((OUT_DIR / "run_metadata.json").read_text(encoding="utf-8"))["status"])
